In [ ]:
# Cell 1: Setup and Initialization
import uuid
from pyspark.sql import functions as F
from datetime import datetime

SSIDGUID = str(uuid.uuid4())
ETLRunTime = datetime.now()
IncrementalExtractTime = datetime.now()
PkgGUID = str(uuid.uuid4())
DJJLastUpdateTime = datetime.now()

In [ ]:
# Cell 2: Define catalog variables and configuration parameters
source_catalog = "{source_catalog}"
target_catalog = "{target_catalog}"
federated_starschema_catalog = "{federated_starschema_catalog}"

MasterSourceSystemName = 'PowerApps'
SourceSubSystemName = ''
SourceDatabaseName = 'PowerApps'
SourceTableName = 'PowerApps'
DestinationSchemaName = 'NUE'
DestinationTableName = 'factCargoAllocationPowerApps'
DestinationDatabaseName = 'DJJStarSchema'
ETLType = 'Incremental'
BatchSize = 1000
PkgName = 'NUE.etl_factCargoAllocationPowerApps'
ETLTemplate = ''

Success = 0
RowsInserted = 0
RowsUpdated = 0
RowsDeleted = 0
ErrorRowCount = 0
TableInitialRowCount = 0
TableFinalRowCount = 0
ETLStatus = ''
ExtractRowCount = 0
SSISGUID = SSIDGUID

### Execute the Initial Metadata Logging

In [ ]:
%run /Workspace/Shared/metadata_framework/metadata_logger

In [ ]:
#Insert/Update records in ETL Master
logger = ETLLogger(spark)

ETLID = logger.log_etl(
    MasterSourceSystemName=MasterSourceSystemName,
    SourceSubSystemName=SourceSubSystemName,
    SourceDatabaseName=SourceDatabaseName,
    SourceTableName=SourceTableName,
    DestinationDatabaseName=DestinationDatabaseName,
    DestinationTableName=DestinationTableName,
    PkgName=PkgName,
    PkgGUID=PkgGUID,
    ETLTemplate=ETLTemplate,
    ETLType=ETLType,
    CheckpointsEnabled=True
)
print(ETLID)

In [ ]:
#Insert records in ETLExecutionLog table
exec_logger = ETLExecutionLogger(spark)
exec_logger.log_execution(
    ETLID=ETLID,
    ETLRunTime=ETLRunTime,
    Success=Success,
    RowsInserted=RowsInserted,
    RowsUpdated=RowsUpdated,
    ErrorRowCount=ErrorRowCount,
    TableInitialRowCount=TableInitialRowCount,
    TableFinalRowCount=TableFinalRowCount,
    ETLStatus=ETLStatus,
    ExtractRowCount=ExtractRowCount,
    SSISGUID=SSIDGUID
)

In [ ]:
ETLExecutionID = spark.sql(f"""SELECT ETLExecutionID from {target_catalog}.metadata.etlexecutionlog where lower(trim(ETLID)) = lower(trim('{ETLID}')) and lower(trim(SSISGUID)) = lower(trim('{SSIDGUID}'))""").first()[0]

### ETL Logic Starts Here

In [ ]:
# Get initial row count of table
TableInitialRowCount = spark.sql(f"SELECT COUNT(1) AS cnt FROM {target_catalog}.NUE.factCargoAllocationPowerApps").first()["cnt"]
print(f"Initial row count: {TableInitialRowCount}")

# Get TargetLastUpdatedDate
target_last_updated_df = spark.sql(f"""
    SELECT COALESCE(MAX(CAST(DJJLastUpdateTime AS DATE)), CAST('1900-01-01' AS DATE)) AS TargetLastUpdatedDate
    FROM {source_catalog}.dbo.ODS_MDS_NUEMillNames
""")
TargetLastUpdatedDate = str(target_last_updated_df.first()["TargetLastUpdatedDate"])
print(f"TargetLastUpdatedDate: {TargetLastUpdatedDate}")

In [ ]:
# Source query: Pull unallocated cargos for Power Apps
# Nue.* -> federated.NUE.*, Brk.* -> federated.djj_brk.*, DJJ.* -> federated.djj.*
# Inline functions fn_DateKeyToDateTime and fn_DateTimeToDateKey called as-is
# ISNULL -> COALESCE, GETDATE() -> current_timestamp()/current_date()
spark.sql(f"""
CREATE TABLE {target_catalog}.temp.UnallocatedCargosInitial AS
SELECT
    d.Fiscal_Week_of_Year AS FiscalWeekKey,
    a.CargoID,
    a.ShippingScenarioID,
    DJJLocationName AS MillName,
    c.EnterpriseGradeGroup AS EnterpriseGrade,
    CAST(a.OrderedWgt / 2240 AS INT) AS OriginalGTs,
    0 AS AllocatedGTs,
    CASE WHEN a.OrderedWgt / 2240 = 0 THEN 0 ELSE a.OrderedDLPricedAmount / (a.OrderedWgt / 2240) END AS CostPerGT,
    CAST({federated_starschema_catalog}.dbo.fn_DateKeyToDateTime(
        CASE WHEN LoadStartDateKey = 19000101 THEN ESTLoadStartDateKey ELSE LoadStartDateKey END) AS DATE) AS LoadStartDate,
    COALESCE(CAST({federated_starschema_catalog}.dbo.fn_DateKeyToDateTime(
        CASE WHEN a.SailDateEstFlag = 1 THEN a.ESTSailDateKey ELSE a.SailDateKey END) AS DATE), CAST('1900-01-01' AS DATE)) AS SailDate,
    COALESCE(CAST({federated_starschema_catalog}.dbo.fn_DateKeyToDateTime(ESTTranDisArrivalDateKey) AS DATE), CAST('1900-01-01' AS DATE)) AS TranDisPortETADate,
    COALESCE(CAST({federated_starschema_catalog}.dbo.fn_DateKeyToDateTime(ESTDischargeFinishDateKey) AS DATE), CAST('1900-01-01' AS DATE)) AS DischargeFinishDate,
    COALESCE(CAST({federated_starschema_catalog}.dbo.fn_DateKeyToDateTime(ESTMINMillTicketDateKey) AS DATE), CAST('1900-01-01' AS DATE)) AS MillTicketInitialArrivalDate,
    COALESCE(CAST({federated_starschema_catalog}.dbo.fn_DateKeyToDateTime(MeltEndDate) AS DATE), CAST('1900-01-01' AS DATE)) AS MeltEndDate,
    COALESCE(h.SupplierName, 'Unknown') AS SupplierName,
    COALESCE(e.PortCountryName, 'Unknown') AS OriginCountry,
    COALESCE(ed.PortName, 'Unknown') AS DestinationPort,
    COALESCE(f.VehicleName, 'TBD') AS Vessel,
    a.StatusDesc,
    0 AS HistoricFlag,
    current_timestamp() AS LastUpdateTime,
    'SQL' AS LastUpdateUser
FROM {federated_starschema_catalog}.NUE.factCargoMillDetails a
LEFT OUTER JOIN {federated_starschema_catalog}.NUE.dimNucorMillLocations b ON a.NucorLocationKey = b.NucorLocationKey
LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimGrade c ON a.BrokerageGradeKey = c.GradeKey
LEFT OUTER JOIN {federated_starschema_catalog}.djj.dimDate d ON d.DateKey = {federated_starschema_catalog}.dbo.fn_DateTimeToDateKey(DATE_ADD(current_date(), -7))
LEFT OUTER JOIN {federated_starschema_catalog}.NUE.dimPorts e ON a.OriginPortKey = e.PortKey
LEFT OUTER JOIN {federated_starschema_catalog}.NUE.dimPorts ed ON a.DestinationPortKey = ed.PortKey
LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.factCargoPurchaseOrder g ON lower(trim(a.ShippingScenarioID)) = lower(trim(g.ShippingScenarioID))
LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimSupplier h ON g.SupplierKey = h.SupplierKey
LEFT OUTER JOIN {federated_starschema_catalog}.djj_brk.dimCargo f ON a.CargoKey = f.CargoKey
WHERE lower(trim(b.DJJLocationName)) = lower(trim('NUCOR UNALLOCATED'))
    AND {federated_starschema_catalog}.dbo.fn_DateKeyToDateTime(ExpirationDateKey) > current_timestamp()
""")
print("Source query completed.")

In [ ]:
# Update HistoricFlag=1 for cargos that have been officially allocated (DJJLocationName IS NULL)
spark.sql(f"""
MERGE INTO {target_catalog}.temp.UnallocatedCargosInitial alpha
USING (
    SELECT a.CargoID, a.ShippingScenarioID, c.DJJLocationName
    FROM (SELECT DISTINCT CargoID, ShippingScenarioID FROM {federated_starschema_catalog}.NUE.factCargoAllocationPowerApps) a
    LEFT OUTER JOIN {federated_starschema_catalog}.NUE.factCargoMillDetails b ON lower(trim(a.ShippingScenarioID)) = lower(trim(b.ShippingScenarioID))
    LEFT OUTER JOIN {federated_starschema_catalog}.NUE.dimNucorMillLocations c ON b.NucorLocationKey = c.NucorLocationKey
) beta ON lower(trim(alpha.CargoID)) = lower(trim(beta.CargoID)) AND lower(trim(alpha.ShippingScenarioID)) = lower(trim(beta.ShippingScenarioID))
AND beta.DJJLocationName IS NULL
WHEN MATCHED THEN UPDATE SET alpha.HistoricFlag = 1
""")
print("HistoricFlag update 1 (allocated cargos) completed.")

In [ ]:
# Update HistoricFlag=1 for cargos where ShippingScenario no longer exists
spark.sql(f"""
MERGE INTO {target_catalog}.temp.UnallocatedCargosInitial alpha
USING (
    SELECT alpha2.CargoID, alpha2.ShippingScenarioID
    FROM {target_catalog}.temp.UnallocatedCargosInitial alpha2
    LEFT OUTER JOIN (
        SELECT a.CargoID, a.ShippingScenarioID
        FROM (SELECT DISTINCT CargoID, ShippingScenarioID FROM {federated_starschema_catalog}.NUE.factCargoAllocationPowerApps) a
        INNER JOIN {federated_starschema_catalog}.NUE.factCargoMillDetails b ON lower(trim(a.ShippingScenarioID)) = lower(trim(b.ShippingScenarioID))
        INNER JOIN {federated_starschema_catalog}.NUE.dimNucorMillLocations c ON b.NucorLocationKey = c.NucorLocationKey
    ) beta ON lower(trim(alpha2.CargoID)) = lower(trim(beta.CargoID)) AND lower(trim(alpha2.ShippingScenarioID)) = lower(trim(beta.ShippingScenarioID))
    WHERE beta.CargoID IS NULL
) src ON lower(trim(alpha.CargoID)) = lower(trim(src.CargoID)) AND lower(trim(alpha.ShippingScenarioID)) = lower(trim(src.ShippingScenarioID))
WHEN MATCHED THEN UPDATE SET alpha.HistoricFlag = 1
""")
print("HistoricFlag update 2 (scenario no longer exists) completed.")

In [ ]:
# UPSERT: Update existing + insert new records
spark.sql(f"""
MERGE INTO {target_catalog}.NUE.factCargoAllocationPowerApps a
USING {target_catalog}.temp.UnallocatedCargosInitial b
ON lower(trim(a.FiscalWeekKey)) = lower(trim(b.FiscalWeekKey))
    AND lower(trim(a.CargoID)) = lower(trim(b.CargoID))
    AND lower(trim(a.ShippingScenarioID)) = lower(trim(b.ShippingScenarioID))
    AND lower(trim(a.EnterpriseGrade)) = lower(trim(b.EnterpriseGrade))
    AND lower(trim(a.MillName)) = lower(trim(b.MillName))
WHEN MATCHED THEN UPDATE SET
    a.OriginalGTs = b.OriginalGTs, a.AllocatedGTs = b.AllocatedGTs, a.CostPerGT = b.CostPerGT,
    a.LoadStartDate = b.LoadStartDate, a.SailDate = b.SailDate, a.TranDisPortETADate = b.TranDisPortETADate,
    a.DischargeFinishDate = b.DischargeFinishDate, a.MillTicketInitialArrivalDate = b.MillTicketInitialArrivalDate,
    a.MeltEndDate = b.MeltEndDate, a.SupplierName = b.SupplierName, a.OriginCountry = b.OriginCountry,
    a.DestinationPort = b.DestinationPort, a.Vessel = b.Vessel, a.HistoricFlag = b.HistoricFlag,
    a.LastUpdateTime = b.LastUpdateTime, a.LastUpdateUser = b.LastUpdateUser
WHEN NOT MATCHED THEN INSERT (
    FiscalWeekKey, CargoID, ShippingScenarioID, MillName, EnterpriseGrade,
    OriginalGTs, AllocatedGTs, CostPerGT, LoadStartDate, SailDate, TranDisPortETADate,
    DischargeFinishDate, MillTicketInitialArrivalDate, MeltEndDate,
    SupplierName, OriginCountry, DestinationPort, Vessel, StatusDesc,
    HistoricFlag, LastUpdateTime, LastUpdateUser
) VALUES (
    b.FiscalWeekKey, b.CargoID, b.ShippingScenarioID, b.MillName, b.EnterpriseGrade,
    b.OriginalGTs, b.AllocatedGTs, b.CostPerGT, b.LoadStartDate, b.SailDate, b.TranDisPortETADate,
    b.DischargeFinishDate, b.MillTicketInitialArrivalDate, b.MeltEndDate,
    b.SupplierName, b.OriginCountry, b.DestinationPort, b.Vessel, b.StatusDesc,
    b.HistoricFlag, b.LastUpdateTime, b.LastUpdateUser
)
""")
print("MERGE INTO factCargoAllocationPowerApps completed.")

In [ ]:
# Update allocations only (where MillName <> 'NUCOR UNALLOCATED')
spark.sql(f"""
MERGE INTO {target_catalog}.NUE.factCargoAllocationPowerApps a
USING {target_catalog}.temp.UnallocatedCargosInitial b
ON lower(trim(a.FiscalWeekKey)) = lower(trim(b.FiscalWeekKey))
    AND lower(trim(a.CargoID)) = lower(trim(b.CargoID))
    AND lower(trim(a.ShippingScenarioID)) = lower(trim(b.ShippingScenarioID))
    AND lower(trim(a.EnterpriseGrade)) = lower(trim(b.EnterpriseGrade))
    AND lower(trim(a.MillName)) <> lower(trim('NUCOR UNALLOCATED'))
WHEN MATCHED THEN UPDATE SET
    a.CostPerGT = b.CostPerGT, a.LoadStartDate = b.LoadStartDate, a.SailDate = b.SailDate,
    a.TranDisPortETADate = b.TranDisPortETADate, a.DischargeFinishDate = b.DischargeFinishDate,
    a.MillTicketInitialArrivalDate = b.MillTicketInitialArrivalDate, a.MeltEndDate = b.MeltEndDate,
    a.SupplierName = b.SupplierName, a.OriginCountry = b.OriginCountry,
    a.DestinationPort = b.DestinationPort, a.Vessel = b.Vessel, a.StatusDesc = b.StatusDesc
""")
print("Allocations-only update completed.")

In [ ]:
# Update HistoricFlag=1 on target for cargos officially allocated (DJJLocationName IS NULL)
spark.sql(f"""
MERGE INTO {target_catalog}.NUE.factCargoAllocationPowerApps alpha
USING (
    SELECT a.CargoID, a.ShippingScenarioID
    FROM (SELECT DISTINCT CargoID, ShippingScenarioID FROM {federated_starschema_catalog}.NUE.factCargoAllocationPowerApps) a
    LEFT OUTER JOIN {federated_starschema_catalog}.NUE.factCargoMillDetails b ON lower(trim(a.ShippingScenarioID)) = lower(trim(b.ShippingScenarioID))
    LEFT OUTER JOIN {federated_starschema_catalog}.NUE.dimNucorMillLocations c ON b.NucorLocationKey = c.NucorLocationKey
    WHERE c.DJJLocationName IS NULL
) beta ON lower(trim(alpha.CargoID)) = lower(trim(beta.CargoID)) AND lower(trim(alpha.ShippingScenarioID)) = lower(trim(beta.ShippingScenarioID))
WHEN MATCHED THEN UPDATE SET alpha.HistoricFlag = 1
""")
print("HistoricFlag update on target completed.")

In [ ]:
# Update StatusDesc for all cargos in case any have changed
spark.sql(f"""
MERGE INTO {target_catalog}.NUE.factCargoAllocationPowerApps alpha
USING {target_catalog}.temp.UnallocatedCargosInitial b
ON lower(trim(alpha.FiscalWeekKey)) = lower(trim(b.FiscalWeekKey))
    AND lower(trim(alpha.CargoID)) = lower(trim(b.CargoID))
    AND lower(trim(alpha.ShippingScenarioID)) = lower(trim(b.ShippingScenarioID))
    AND lower(trim(alpha.EnterpriseGrade)) = lower(trim(b.EnterpriseGrade))
WHEN MATCHED THEN UPDATE SET alpha.StatusDesc = b.StatusDesc
""")
print("StatusDesc update completed.")

### Close Out the ETL - Final Audit & Logging

In [ ]:
# Get final rowcount of target table
final_count_row = spark.sql(f"""SELECT COUNT(1) AS cnt FROM {target_catalog}.NUE.factCargoAllocationPowerApps""").first()
TableFinalRowCount = final_count_row['cnt'] if final_count_row else 0
print(f"Final row count: {TableFinalRowCount}")

In [ ]:
# Update records in ETLExecutionLog table with success status
Success = 1
exec_logger.log_execution(
    ETLID=ETLID,
    ETLRunTime=ETLRunTime,
    Success=Success,
    RowsInserted=RowsInserted,
    RowsUpdated=RowsUpdated,
    ErrorRowCount=ErrorRowCount,
    TableInitialRowCount=TableInitialRowCount,
    TableFinalRowCount=TableFinalRowCount,
    ETLStatus=ETLStatus,
    ExtractRowCount=ExtractRowCount,
    SSISGUID=SSIDGUID
)

In [ ]:
# Update ETLMaster with final metadata
spark.sql(f"""UPDATE {target_catalog}.metadata.ETLMaster
    SET IncrementalExtractTime = '{IncrementalExtractTime}',
        ETLTemplate = '{ETLTemplate}',
        ETLType = '{ETLType}'
    WHERE ETLID = '{ETLID}'""")

In [ ]:
# Cleanup - drop temp table
spark.sql(f"DROP TABLE IF EXISTS {target_catalog}.temp.UnallocatedCargosInitial")
print("Temp table cleaned up. ETL completed successfully.")